In [ ]:
!pip install langchain-core
import os, re
import tiktoken
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

d:\Astro Books\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5

In [ ]:
!pip install langchain-core


Note: you may need to restart the kernel to use updated packages.


In [10]:
import re
import os

BOOK_NAMES = {
    "BPHS_English_Book.md":   "Brihat Parashara Hora Shastra (English)",
    "BPHS_Hindi_Book.md":     "Brihat Parashara Hora Shastra (Hindi)",
    "Phaladeepika_part_1.md": "Phaladeepika Part 1",
    "Phaladeepika_part_2.md": "Phaladeepika Part 2",
    "Saravali.md":            "Saravali",
}

def split_by_chapters(text):
    pattern = r'(?=# Chapter \d+)'
    chunks = re.split(pattern, text)
    return [c.strip() for c in chunks if len(c.strip()) > 200]

def remove_toc(text: str) -> str:
    # find the first real "# Chapter" and start from there
    match = re.search(r'\n# Chapter', text)
    if match:
        return text[match.start():]
    return text

def split_documents(docs):
    final_chunks = []

    for doc in docs:
        filename = os.path.basename(doc.metadata.get("source", ""))
        book_name = BOOK_NAMES.get(filename, filename)

        # normalize line endings
        clean_text = doc.page_content.replace('\r\n', '\n').replace('\r', '\n')
        clean_text = remove_toc(clean_text)

        # stage 1: split by chapter
        chapter_chunks = split_by_chapters(clean_text)

        for chapter_text in chapter_chunks:
            # extract chapter title from first line
            first_line = chapter_text.split('\n')[0].strip()
            chapter_title = first_line.replace('#', '').strip()

            from langchain_core.documents import Document
            chunk_doc = Document(
                page_content=chapter_text,
                metadata={
                    "book": book_name,
                    "chapter": chapter_title,
                    "source": doc.metadata.get("source", "")
                }
            )

            # stage 2: re-split if too large
            # in your chapter loop, skip tiny chunks# in your chapter loop, skip tiny chunks
            if len(chapter_text.strip()) < 300:
                continue
            if len(chapter_text) > 700:
                sub_chunks = char_splitter.split_documents([chunk_doc])
                final_chunks.extend(sub_chunks)
            else:
                final_chunks.append(chunk_doc)

    print(f'Total chunks created: {len(final_chunks)}')
    return final_chunks

In [ ]:
# test it quickly
from src.document_loader import load_documents

docs = load_documents()

chunks = split_documents(docs)
print(f"Total: {len(chunks)}")

# check first 20 chunks
for i, c in enumerate(chunks[:20]):
    print(f"\n[{i}] {c.metadata}")
    print(c.page_content[:150])

Document 5 Loaded
Total chunks created: 5272
Total: 5272

[0] {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 5', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
# Chapter 5

The method given by astrologers can help to foretell the exact moment of birth from the time of coition. The query has to be very well ti

[1] {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 8', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
# Chapter 8

What is called Jaimini deals with aspects Rasi basis. Maharshi Jaimini was the sage who expanded Parasari principles very extensively on 

[2] {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 24', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
# Chapter 24

The chapter is again one of the most important chapters. Here, various effects of a lord being in a specific Bhava are fully covered. Ho

[3] {'book': 'Brihat Parashara Hora Shast

In [12]:
import random

for c in random.sample(chunks, 20):
    print(c.metadata)
    print(c.page_content[:250])
    print("-"*80)

{'book': 'Brihat Parashara Hora Shastra (Hindi)', 'chapter': 'भूमिका', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_Hindi_Book.md'}
७ | 
भ्रष्ट | 
७ | 
८ | 
९ | 
१० | 
११ | 
१२ | 
१ | 
२ | 
३ | 
४ | 
५ | 
६ | 
 | 
सौम्य | 
३।३० | 

८ | 
कुलघ्न | 
८ | 
९ | 
१० | 
११ | 
१२ | 
१ | 
२ | 
३ | 
४ | 
५ | 
६ | 
७ | 
 | 
निर्मल | 
४।० | 

९ | 
गरल | 
९ | 
१० | 
११ | 
१२ | 
१ | 
२ | 
३ | 

--------------------------------------------------------------------------------
{'book': 'Phaladeepika Part 2', 'chapter': 'ADHYAYA XX.', 'source': 'D:\\Astro Books\\Books\\md Files\\Phaladeepika_part_2.md'}
Sloka 11. - Pain in the head, belly-ache, trouble in the anus, doing agricultural operations, loss of house, wealth and corn, sickness to children and wife in an intense form - all these will occur when the Bhukti of Venus in the Sun's Mahadasa is in
--------------------------------------------------------------------------------
{'book': 'Brihat Parashara Hora Shastra (Hindi)', 'chapter': 'भूमिका', 

In [13]:
for i,c in enumerate(chunks):
    print(
        i,
        len(c.page_content),
        c.metadata
    )

0 1591 {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 5', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
1 875 {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 8', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
2 1030 {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 24', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
3 1364 {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 34', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
4 2093 {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 34', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
5 1371 {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 34', 'source': 'D:\\Astro Books\\Books\\md Files\\BPHS_English_Book.md'}
6 1334 {'book': 'Brihat Parashara Hora Shastra (English)', 'chapter': 'Chapter 34', 'source

In [ ]:
\

Response(id='resp_07b1cc9922ca7016006a40e509c61c8190bd4ec21cb27287f4', created_at=1782637833.0, error=None, incomplete_details=None, instructions='You are a helpful assistant.', metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_07b1cc9922ca7016006a40e50b55808190925955ca903ca199', content=[ResponseOutputText(annotations=[], text='The term "OpenAI syntax" isn\'t a standard concept, but it might refer to the way in which you interact with OpenAI\'s models, like how to structure prompts or queries. When working with OpenAI\'s models, here are some useful guidelines:\n\n1. **Clarity**: Be clear and specific in your instructions. For example, instead of saying "Tell me about dogs," you might say, "Explain the different breeds of dogs and their characteristics."\n\n2. **Context**: Provide context if needed. If you\'re asking a complex question, a bit of background information can help the model give you a more accurate response.\n\n3. **Form